## Multi-arm Environment with mjSpec
- spawn multiple UR5e using mjSpec

In [23]:
import mujoco as mj
import mujoco_viewer

import numpy as np
import time

import xml.etree.ElementTree as ET
from lxml import etree

In [ ]:
def print_xml(xml_input,color=True):
    if isinstance(xml_input, ET.Element):
        rough_string = ET.tostring(xml_input, encoding='unicode')
    else:
        rough_string = xml_input

    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.fromstring(rough_string, parser=parser)
    pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
    print(pretty_xml)

def save_spec_as_xml(spec, file_path):
    xml_string = spec.to_xml_string()
    with open(file_path, 'w') as f:
        f.write(xml_string)

### Synthesize mjSpec scene
- load default floor
- add frame & attach UR robot

In [ ]:
path = '../asset/floor_white_gray.xml'
spec = mj.MjSpec.from_file(path)

In [ ]:
multi_robot_config = {
    "frame": {
        "frame1": {
            "name": "UR_1",
            "pos": [1, 0, 0],
            "quat": [0.7071, 0, 0, 0.7071] # wxyz order
        },
        "frame2": {
            "name": "UR_2",
            "pos": [0, 1, 0],
            "quat": [0, 0, 0, 1]
        },
        "frame3": {
            "name": "UR_3",
            "pos": [-1, 0, 0],
            "quat": [0.7071, 0, 0, -0.7071]
        },
        "frame4": {
            "name": "UR_4",
            "pos": [0, -1, 0],
            "quat": [1, 0, 0, 0]
        }
    },
    
    "robot_file": "../assets/panda/robot.xml"
}

In [27]:
# attach frames
frame_list = []
for key, info in multi_robot_config["frame"].items():
    frame = spec.worldbody.add_frame(pos=info["pos"], quat=info["quat"])
    frame_list.append(frame)

In [ ]:
# load robot spec
for i, frame in enumerate(frame_list):
    spec_arm = mj.MjSpec.from_file(multi_robot_config["robot_file"])
    frame.attach_body(spec_arm.body("base"), 'attached-', f'-{i}')

In [29]:
model = spec.compile()
data = mj.MjData(model)

In [30]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# data reset
mj.mj_resetData(model, data)

while True:
    if viewer.is_alive:
        
        mj.mj_step(model, data)
        viewer.render()

    else:
        break

# close
viewer.close()

Pressed ESC
Quitting.


In [31]:
print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <default>
    <default class="attached-main-0"/>
    <default class="attached-main-1"/>
    <default class="attached-main-2"/>
    <default class="attached-main-3"/>
  </default>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
    <material name="attached-Face636_001-0" shininess="0.45" rgba="0.901961 0.921569 0.929412 1"/>
    <material name="attached-Part__Feature017_001-0" shininess="0.45"/>
    <material name="attached-Part__Feature018_001-0" shinines

In [32]:
def get_actuator_name (model, data):
    control_names = [mj.mj_id2name(model,mj.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
    return control_names

actuator_names = get_actuator_name(model,data) # same naming convention - prefix and postfix

In [33]:
model.nu

28